<a href="https://colab.research.google.com/github/jellyandcream494/Transformer-Component-Selection/blob/main/nanoGPT_m2_modularattention_wikitext2_ultimatecombinator_fromcolab_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NanoGPT on Mac M2 (End-to-End)
This notebook pulls all necessary code from `nanoGPT` to run a miniature character-level Shakespeare model directly on a Macbook Air M2.
We use the configuration overrides requested: `--device=mps --compile=False --eval_iters=20 --log_interval=1 --block_size=64 --batch_size=12 --n_layer=4 --n_head=4 --n_embd=128 --max_iters=2000 --lr_decay_iters=2000 --dropout=0.0`.

## Step 2: Prepare the Data
This cell downloads the WikiText-2 dataset and encodes it into `train.bin` and `val.bin` using subword tokenization (BPE) via OpenAI's `tiktoken`.

In [1]:
from google.colab import drive
import pandas as pd
import os
import time

# Mount Google Drive to access your folders
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import pickle
import numpy as np
import requests
import tiktoken

# We'll save the data to a local 'data/wikitext2_bpe' directory to keep things organized.
data_dir = os.path.join('data', 'wikitext2_bpe')
os.makedirs(data_dir, exist_ok=True)

input_file_path = os.path.join(data_dir, 'wikitext-2-train.txt')
if not os.path.exists(input_file_path):
    print("Downloading WikiText-2 dataset...")
    data_url = 'https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt'
    with open(input_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
print(f"length of dataset in characters: {len(data):,}")

# Use tiktoken for subword tokenization
enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(data, allowed_special={"<|endoftext|>"})
print(f"Total tokens: {len(tokens):,}")

# Split into train/val (e.g., 90/10)
n = len(tokens)
train_ids = tokens[:int(n*0.9)]
val_ids = tokens[int(n*0.9):]
print(f"train has {len(train_ids):,} tokens")
print(f"val has {len(val_ids):,} tokens")

# Export to bin files
train_ids = np.array(train_ids, dtype=np.uint16)
val_ids = np.array(val_ids, dtype=np.uint16)
train_ids.tofile(os.path.join(data_dir, 'train.bin'))
val_ids.tofile(os.path.join(data_dir, 'val.bin'))

# Save meta.pkl
vocab_size = enc.n_vocab # usually 50257
meta = {'vocab_size': vocab_size}
with open(os.path.join(data_dir, 'meta.pkl'), 'wb') as f:
    pickle.dump(meta, f)
print("Data preparation complete.")

length of dataset in characters: 10,780,437
Total tokens: 2,448,382
train has 2,203,543 tokens
val has 244,839 tokens
Data preparation complete.


## Step 3: Model Architecture
This contains the full `model.py` logic exactly as is. We include it directly in the notebook as requested so it's fully self-contained.

In [3]:
"""
Full definition of a GPT Language Model, all of it in this single file.
References:
1) the official GPT-2 TensorFlow implementation released by OpenAI:
https://github.com/openai/gpt-2/blob/master/src/model.py
2) huggingface/transformers PyTorch implementation:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
"""

import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        # MODULAR: Instantiate the attention layer passed inside the configuration
        self.attn = config.attention_class(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster
    # MODULAR: Attach the specific class reference
    attention_class: type = CausalSelfAttention

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {} # default to empty dict
        # only dropout can be overridden see more notes below
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        # we can override the dropout rate, if desired
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

## Step 4: Training Setup & Loop
This is the `train.py` code, with the following **alterations** to make it run end-to-end on an M2 Mac in a Jupyter Notebook:
- **Deletion:** Removed `exec(open('configurator.py').read())` and explicitly declared hyperparameters here to match your custom command.
- **Alteration:** Set `dataset = 'wikitext2_bpe'` to point to the local dataset we prepared.
- **Alteration:** Forced device-specific configurations (`device = 'mps'`, `compile = False`) specifically for Mac M2 support.
- **Deletion:** Removed DDP (Distributed Data Parallel) blocks (`RANK`, `init_process_group`, `DistributedDataParallel as DDP`).
- **Alteration:** Adjusted optimizer/scaler backpass specifically preventing CUDA requirements.

In [4]:
import os
import time
import math
import pickle
from contextlib import nullcontext
import numpy as np
import torch
import traceback
import tiktoken

# -----------------------------------------------------------------------------
# Global Training Configuration (pulled from standard nanoGPT setups)
out_dir = 'out-wikitext2-bpe'
eval_interval = 250
log_interval = 250
eval_iters = 20
always_save_checkpoint = False

dataset = 'wikitext2_bpe'
gradient_accumulation_steps = 1

# Default Model configuration (used directly if not overridden)
n_layer = 4
n_head = 4
n_embd = 128
dropout = 0.0
bias = False

learning_rate = 1e-3
max_iters = 2000
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.99
grad_clip = 1.0

decay_lr = True
warmup_iters = 100
lr_decay_iters = 2000
min_lr = 1e-4

if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

dtype = 'float16'
compile = False
# -----------------------------------------------------------------------------

# Device configuration securely
torch.manual_seed(1337)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device_type = 'mps' if 'mps' in device else ('cuda' if 'cuda' in device else 'cpu')
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type in ['cpu', 'mps'] else torch.amp.autocast(device_type=device_type, dtype=ptdtype)


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


def train_model(model_name, attention_class, block_size=64, batch_size=12, data_slice_fraction=1.0, n_layer=4, n_head=4, n_embd=128, temperature=1.0):
    print(f"\n--- Training nanoGPT with {model_name} (Layers: {n_layer}, Block: {block_size}, Batch: {batch_size}, Dataset %: {data_slice_fraction*100}%, Temp: {temperature}) ---")

    # Track RAM usage
    if device_type == 'mps':
        torch.mps.empty_cache()
    elif device_type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # Dynamic batch getter
    def get_batch(split):
        data_dir = os.path.join('data', dataset)
        if split == 'train':
            data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
        else:
            data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')

        # Restrict the data indexing boundary based on our fraction
        max_idx = int(len(data) * data_slice_fraction)
        if max_idx <= block_size:
            max_idx = block_size + 1 # Fallback for safety limit

        ix = torch.randint(max_idx - block_size, (batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

        if device_type == 'cuda':
            x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
        else:
            x, y = x.to(device), y.to(device)
        return x, y

    meta_vocab_size = None
    meta_path = os.path.join('data', dataset, 'meta.pkl')
    if os.path.exists(meta_path):
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        meta_vocab_size = meta['vocab_size']

    # Initialize model using our module-injected config
    model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,
                      bias=bias, vocab_size=None, dropout=dropout, attention_class=attention_class)

    model_args['vocab_size'] = meta_vocab_size if meta_vocab_size is not None else 50304
    gptconf = GPTConfig(**model_args)
    model = GPT(gptconf)
    model.to(device)

    scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16' and device_type == 'cuda'))
    optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        model.train()
        return out

    best_val_loss = 1e9

    try:
        X, Y = get_batch('train')
        t0 = time.time()
        start_time = t0

        for iter_num in range(max_iters + 1):
            lr = get_lr(iter_num) if decay_lr else learning_rate
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

            if iter_num % eval_interval == 0:
                losses = estimate_loss()
                print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
                val_loss = losses['val'].item()
                if val_loss < best_val_loss:
                    best_val_loss = val_loss

            for micro_step in range(gradient_accumulation_steps):
                with ctx:
                    logits, loss = model(X, Y)
                    loss = loss / gradient_accumulation_steps

                X, Y = get_batch('train')

                if device_type == 'cuda' and dtype == 'float16':
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

            if grad_clip != 0.0:
                if device_type == 'cuda' and dtype == 'float16':
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            if device_type == 'cuda' and dtype == 'float16':
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

        end_time = time.time()
        execution_time = end_time - start_time

        # Calculate GPU RAM allocated safely
        try:
            if device_type == 'mps':
                mem_mb = torch.mps.current_allocated_memory() / (1024 ** 2)
            elif device_type == 'cuda':
                mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            else:
                mem_mb = 0.0
        except Exception:
            mem_mb = 0.0

        print("\nGenerating sample text for multiple prompts...")
        model.eval()

        prompts = [
            "Born in 1892, he was an American",
            "During the Second World War, the military",
            "The film received mixed reviews from critics, but",
            "The species is native to"
        ]

        enc = tiktoken.get_encoding("gpt2")
        encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
        decode = lambda l: enc.decode(l)

        total_inference_time = 0.0
        tokens_to_generate = 150
        generated_texts = {}

        for i, prompt in enumerate(prompts):
            print(f"\n--- Prompt: '{prompt}' ---")
            start_ids = encode(prompt)
            x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

            inf_t0 = time.time()
            with torch.no_grad():
                with ctx:
                    out_ids = model.generate(x, tokens_to_generate, temperature=temperature, top_k=200)
            inf_t1 = time.time()

            total_inference_time += (inf_t1 - inf_t0)
            gen_text = decode(out_ids[0].tolist())
            print(gen_text)
            generated_texts[f"Prompt {i+1}"] = gen_text

        print("="*40)

        tokens_per_sec = (tokens_to_generate * len(prompts)) / total_inference_time if total_inference_time > 0 else 0

        perplexity = math.exp(best_val_loss)
        result_dict = {
            "Model": model_name,
            "Layers": n_layer,
            "Heads": n_head,
            "Embd": n_embd,
            "Temperature": temperature,
            "Val Loss": best_val_loss,
            "Perplexity": perplexity,
            "Parameters (M)": round(model.get_num_params()/1e6, 2),
            "GPU RAM (MB)": mem_mb,
            "Block Size": block_size,
            "Batch Size": batch_size,
            "Data %": data_slice_fraction,
            "Time (s)": execution_time,
            "Inference (tok/s)": tokens_per_sec
        }
        result_dict.update(generated_texts)
        return result_dict

    except RuntimeError as e:
        # Catch OOM Memory limits dynamically
        error_str = str(e).lower()
        if "out of memory" in error_str or "memory" in error_str or "allocat" in error_str:
            print(f"!!! MEMORY LIMIT REACHED (OOM) !!! Model: {model_name} (Block: {block_size}, Batch: {batch_size})")
            print(f"Skipping parameter combination. Trace: \n{error_str[:150]}")

            # Clean up properly to avoid cascading OOMs
            del model
            if device_type == 'mps':
                torch.mps.empty_cache()
            elif device_type == 'cuda':
                torch.cuda.empty_cache()

            return {
                "Model": model_name,
                "Layers": n_layer,
                "Heads": n_head,
                "Embd": n_embd,
                "Temperature": temperature,
                "Val Loss": "OOM",
                "Perplexity": "OOM",
                "Parameters (M)": "OOM",
                "GPU RAM (MB)": "OOM",
                "Block Size": block_size,
                "Batch Size": batch_size,
                "Data %": data_slice_fraction,
                "Time (s)": "OOM",
                "Inference (tok/s)": "OOM",
                "Prompt 1": "OOM",
                "Prompt 2": "OOM",
                "Prompt 3": "OOM",
                "Prompt 4": "OOM"
            }
        else:
            raise e

In [5]:
# -----------------------------------------------------------------------------
# EXPERIMENT 1: CUSTOM ATTENTION MECHANISMS
# -----------------------------------------------------------------------------
# Note: The baseline `CausalSelfAttention` is already a standard Multi-Head
# Attention (MHA) implementation, but it utilizes PyTorch 2.0's Flash Attention.
# To give you a fair comparison of architectural overhead, we'll implement a
# purely manual `VanillaMultiHeadAttention` (no flash) and test it alongside
# the requested memory-bandwidth optimized `MultiQueryAttention` (MQA).

class VanillaMultiHeadAttention(nn.Module):
    """Unoptimized MHA to demonstrate the speed differential of standard attention."""
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MultiQueryAttention(nn.Module):
    """MQA: All heads share the exact same Key and Value projection to save memory bandwidth."""
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.dropout = config.dropout

        self.c_q = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.c_kv = nn.Linear(config.n_embd, 2 * self.head_dim, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()

        q = self.c_q(x)
        k, v = self.c_kv(x).split(self.head_dim, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, 1, self.head_dim).transpose(1, 2)
        v = v.view(B, T, 1, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class LinearAttention(nn.Module):
    """Linear Attention: replaces softmax with a simple normalization or activation, calculating relations efficiently."""
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q = F.relu(q)
        k = F.relu(k)

        att = (q @ k.transpose(-2, -1))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, 0.0)
        att = att / (att.sum(dim=-1, keepdim=True) + 1e-6)

        att = self.attn_dropout(att)
        y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

# -----------------------------------------------------------------------------
# MASS EXPERIMENT RUNNER
# -----------------------------------------------------------------------------
results_registry = []

attention_architectures = [
    ("Baseline (Causal Self Attention with PyTorch 2.0 and Flash)", CausalSelfAttention),
    ("Vanilla MHA", VanillaMultiHeadAttention),
    ("Multi-Query Attention", MultiQueryAttention),
    ("Linear Attention", LinearAttention)
]

# We include very small, normal, and very large configurations
# Scaling test dimensions
experiment_blocks = [4096]
experiment_batches = [2] # Very large batch (64) is bound to push hardware limits
experiment_data_fractions = [0.1]
experiment_temperatures = [0.8]

# Introduce architectural depth & width combinations (n_layer, n_head, n_embd)
# Ensuring that n_embd % n_head == 0!
experiment_structures = [
    # (2, 2, 64),
    (4, 4, 128)
    # (8, 8, 256)

]

print("Executing automated modular parameter sweeps...")
experiment_id = 1
for struct in experiment_structures:
    n_layer, n_head, n_embd = struct
    for data_frac in experiment_data_fractions:
        for block in experiment_blocks:
            for batch in experiment_batches:
                for temp in experiment_temperatures:
                    for arch_name, arch_class in attention_architectures:
                        # Runs securely catching any Out Of Memory exceptions dynamically
                        result = train_model(
                            model_name=arch_name,
                            attention_class=arch_class,
                            block_size=block,
                            batch_size=batch,
                            data_slice_fraction=data_frac,
                            n_layer=n_layer,
                            n_head=n_head,
                            n_embd=n_embd,
                            temperature=temp
                        )
                        result["Experiment Index"] = experiment_id
                        experiment_id += 1
                        results_registry.append(result)

print("\n--- All experiments concluded safely! Check constraints below! ---")

Executing automated modular parameter sweeps...

--- Training nanoGPT with Baseline (Causal Self Attention with PyTorch 2.0 and Flash) (Layers: 4, Block: 4096, Batch: 2, Dataset %: 10.0%, Temp: 0.8) ---
number of parameters: 7.22M


/tmp/ipykernel_12888/2110755826.py:122: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16' and device_type == 'cuda'))


num decayed parameter tensors: 18, with 7,743,616 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True
step 0: train loss 10.8514, val loss 10.8458
step 250: train loss 5.1581, val loss 6.2880
step 500: train loss 4.3895, val loss 6.5236
step 750: train loss 3.7440, val loss 6.6160
step 1000: train loss 3.4810, val loss 6.9558
step 1250: train loss 3.2437, val loss 7.1597
step 1500: train loss 3.0597, val loss 7.2478
step 1750: train loss 2.9727, val loss 7.2937
step 2000: train loss 2.8641, val loss 7.3940

Generating sample text for multiple prompts...

--- Prompt: 'Born in 1892, he was an American' ---
Born in 1892, he was an American version of the Year , although they have a severe rains and his career in that " . He made a " 
 
 = = = = 
 
 
 
 
 
 
 In 2010 , the West All of the source of the region , but will be the 1970 ) , the top of the hands , <unk> <unk> , the Union <unk> ) 
 On the 1950 
 
 
 The Academy . He became the city 's pr

/tmp/ipykernel_12888/2110755826.py:122: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16' and device_type == 'cuda'))


step 0: train loss 10.8338, val loss 10.8293
step 250: train loss 5.1974, val loss 6.2618
step 500: train loss 4.3393, val loss 6.4962
step 750: train loss 3.7797, val loss 6.6875
step 1000: train loss 3.4904, val loss 6.8180
step 1250: train loss 3.2690, val loss 7.2176
step 1500: train loss 3.1307, val loss 7.3535
step 1750: train loss 2.9338, val loss 7.5063
step 2000: train loss 2.8881, val loss 7.6239

Generating sample text for multiple prompts...

--- Prompt: 'Born in 1892, he was an American' ---
Born in 1892, he was an American captain and had the <unk> on 22 June 21 June 2012 , the " . 
 On 20 , he was named it to the Central Australia , two cultural and in Europe . 
 
 
 
 
 = = = New York City = 
 = 
 
 
 The synagogue , the process = = = = History = 
 
 
 = = = 
 
 
 <unk> , the following year , securing over 100 miles ( 6 m3 − 
 
 
 
 
 The Jews resulted in the region of them on hold a member of the East Coast of the hurricane made using various forces of the storm fell a

## Step 5: Evaluate Architecture Results
The models have been trained and generation has been tested inline. Displaying the baseline statistics.

In [6]:
import pandas as pd
from IPython.display import display

def display_results(results):
    df = pd.DataFrame(results)

    # Format the numerical columns for readability while bypassing missing features via errors="ignore"
    for col in ["Val Loss", "Perplexity", "Time (s)", "GPU RAM (MB)", "Inference (tok/s)"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # format safely if numeric, else keep OOM
            df[col] = df[col].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "OOM")

    df["Parameters (M)"] = pd.to_numeric(df["Parameters (M)"], errors='coerce')
    df["Parameters (M)"] = df["Parameters (M)"].apply(lambda x: f"{x:.2f}" if pd.notnull(x) else "OOM")

    # Sort successfully finished items by execution time
    df_success = df[df["Time (s)"] != "OOM"].copy()
    df_failed = df[df["Time (s)"] == "OOM"].copy()

    # Ensure they sort logically
    if not df_success.empty:
        df_success["Sort_Key"] = pd.to_numeric(df_success["Time (s)"])
        df_success = df_success.sort_values(by="Sort_Key").drop("Sort_Key", axis=1)

    df_combined = pd.concat([df_success, df_failed]).reset_index(drop=True)

    # Reorder columns to display intuitively, including new Prompts columns and Experiment Index
    cols = ["Experiment Index", "Model", "Layers", "Heads", "Embd", "Temperature", "Data %", "Block Size", "Batch Size", "Parameters (M)", "Time (s)", "Inference (tok/s)", "GPU RAM (MB)", "Val Loss", "Perplexity", "Prompt 1", "Prompt 2", "Prompt 3", "Prompt 4"]
    df_combined = df_combined[[c for c in cols if c in df_combined.columns]]

    print("\n================ FULL MASS EXPERIMENT RESULTS ================\n")
    # Setting max rows high so the notebook doesn't truncate the middle experiments
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 50):
        display(df_combined)

display_results(results_registry)


================ FULL MASS EXPERIMENT RESULTS ================



,Experiment Index,Model,Layers,Heads,Embd,Temperature,Data %,Block Size,Batch Size,Parameters (M),Time (s),Inference (tok/s),GPU RAM (MB),Val Loss,Perplexity,Prompt 1,Prompt 2,Prompt 3,Prompt 4
0,1,Baseline (Causal Self Attention with PyTorch 2...,4,4,128,0.8,0.1,4096,2,7.22,247.2046,321.2691,4982.3403,6.2880,538.0882,"Born in 1892, he was an American version of th...","During the Second World War, the military stud...","The film received mixed reviews from critics, ...","The species is native to the <unk> , and indiv..."
1,3,Multi-Query Attention,4,4,128,0.8,0.1,4096,2,7.12,613.8672,229.1254,8380.5298,6.3478,571.2328,"Born in 1892, he was an American Civil War . W...","During the Second World War, the military and ...","The film received mixed reviews from critics, ...",The species is native to the main application ...
2,2,Vanilla MHA,4,4,128,0.8,0.1,4096,2,7.22,614.0092,262.7431,8373.1060,6.2618,524.1467,"Born in 1892, he was an American captain and h...","During the Second World War, the military educ...","The film received mixed reviews from critics, ...",The species is native to be available to the s...
3,4,Linear Attention,4,4,128,0.8,0.1,4096,2,7.22,772.4211,260.5633,7367.0747,6.3197,555.3848,"Born in 1892, he was an American global reserv...","During the Second World War, the military in w...","The film received mixed reviews from critics, ...","The species is native to <unk> , which might h..."


In [7]:
# Note: Colab cannot automatically detect the notebook's current Drive path.
# Change this variable to match the specific folder path in your Drive.
drive_folder = '/content/drive/MyDrive/MSc_AI/Unit9_ADL/Unit9_Assignment2/'
os.makedirs(drive_folder, exist_ok=True)

try:
    if 'results_registry' in globals() and results_registry:
        df_results = pd.DataFrame(results_registry)
        ts = time.strftime('%Y%m%d-%H%M%S')
        out_path = os.path.join(drive_folder, f'results_{ts}.csv')
        df_results.to_csv(out_path, index=False)
        print(f"Results successfully saved to Google Drive at: {out_path}")
    else:
        print('No results found in `results_registry` to save.')
except Exception as e:
    print('Failed to save results:', e)

Results successfully saved to Google Drive at: /content/drive/MyDrive/MSc_AI/Unit9_ADL/Unit9_Assignment2/results_20260624-111801.csv
